# K513 · Week 3 Homework
## Three tests, and knowing which number to believe

This assignment is Tuesday's session applied to a table you already know. You will run all three
tests, and then spend most of your writing on the two results that should **not** go in a report —
one because the chart contradicts it, and one because the test's own assumptions are strained.

**The dataset.** `world_indicators_2010.csv` — 208 countries, 24 columns, one row per country. The
same table you drew last week.

**The story.** You are still at the foundation. The program is going ahead, and a health ministry has
asked you the question everybody eventually asks: *does spending more per person buy longer lives?*

**Where the answers go.** Not in this notebook — in the Canvas quiz **Week 3 Homework - Relationships Between Variables**. Work them out here,
then write them there. Nothing typed into this notebook is collected, so an answer left here scores
zero.

**Due Sunday 11:59 pm.** Submit the Canvas quiz, and paste a link to this notebook into its first
question.

---
### Before you type anything

**File → Save a copy in Drive.**

This notebook is read-only for you. You can type into it and run it and it will look completely
normal, but nothing you do will be saved. Save your own copy first, every time.

---

### Using AI in this notebook

Gemini is built into Colab and you are welcome to use it here. Two things worth knowing:

- It does not know which columns you have or what we covered in class. Whatever it writes, you own.
- The most useful thing you can ask it is **"explain what this line does"** — not "write it for me".

AI will also write code cells out of order and leave redundant ones behind. **Clean the notebook
before you submit it.** A notebook that only runs top to bottom if you know which cells to skip is
not a finished piece of work.

The specific trap this week is that an AI will happily run a test and report a p-value with
no way of knowing whether that test was the right one for your two columns — because that depends on
the shape of your data, which it cannot see. Worse, it will not tell you when a significant result is
being produced by two rows. Part 2 has one of those in it.

---

### Turn off Unwanted AI Assistance

AI-powered coding completion is turned on by default. It is convenient but does not give you a chance
to think and learn. Turning it off helps you learn. You can always turn it back on when needed.
- Tools → settings → AI Assistance → Uncheck "Show AI-powered inline code completions"
- Tools → settings → Uncheck "Show context-powered code completions"

---

### How to run a cell

Click on a cell, then press **Shift + Enter**. That runs it and moves you to the next one. If
anything ever looks wrong: **Runtime → Restart session and run all**.


---
## 0 · Setup

Run all four cells. Nothing here needs changing.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr
from scipy.stats import f_oneway, chi2_contingency

pd.set_option('display.precision', 3)
pd.set_option('display.max_columns', None)

In [ ]:
WORLD_URL = "https://raw.githubusercontent.com/jl-uscn/k513-data/main/world_indicators_2010.csv"

world_df = pd.read_csv(WORLD_URL)
world_df.shape

The tests in `scipy.stats` will not accept a column with missing values (NaN) in it. Three of the four columns you
need here have missing values — `Region` is complete — so the cell below drops any country that is missing one
of the other three.

**Everything in Parts 1 and 3 uses `clean_df`, not `world_df`.** Part 2 goes back to `world_df` on
purpose, and says so when it does.

Look at the shape it prints. You are about to run tests on a table with fewer countries in it than
the one you started with, and the number you dropped belongs in any report you write from it — it is
one of the limitations the Analyst's Note in Part 4 asks for. **Canvas question 2** asks for it.

*(This is the NaN handling on the Reference slides at the back of Tuesday's deck.)*


In [ ]:
clean_df = world_df[['Region', 'Population Urban %',
                     'Life Expectancy Female', 'Health Exp/Capita']].dropna()
clean_df.shape

---
# Part 1 · Two numbers

**How to choose a test** *(taught in Week 3 Session 1)*. Read it in column order: what you **have** is a
fact about your data, not a choice. What you **want** is the second column. Only then can the appropriate test be chosen.

| What you have | Your question | The test | And always draw |
|---|---|---|---|
| two numbers, both symmetric | do they move together? | `pearsonr()` | `scatterplot()` |
| two numbers, one skewed | do they move together? | `spearmanr()` | `scatterplot()` |
| one number, one category | is it different by group? | `f_oneway()` | `boxplot(x=cat, y=num)` |
| two categories | are the two related? | `chi2_contingency()` | `countplot(x=, hue=)` |

**The skew rule.** Skew first, correlate second.

| `skew()` | |
|---|---|
| between −1 and 1 | close enough to symmetric → `pearsonr()` |
| outside −1 and 1 | skewed → `spearmanr()` |
| **either column fails** | **the pair fails** |

Every test this week runs in the same order, and this notebook refers to the steps by number:

- **Step 0** — `skew()`, which decides Pearson or Spearman before you correlate anything.
- **Step 1** — the chart.
- **Step 2** — the test. It never overrules the chart.

## 1.1 · Step 0

Before you correlate anything, ask how skewed each column is. This decides which coefficient you are
allowed to use, and it is not a judgment call.


In [ ]:
clean_df.skew(numeric_only=True)

## 1.2 · The first pair: urbanization and how long women live

Both of these columns passed the skew rule (see the cell above: if the skewness of both variables is in the range −1 to 1, use Pearson), so this pair is straightforward. Pass the two columns **in order**, separated by a comma: `pearsonr(first_column, second_column)`.

Python also lets you label each input — `pearsonr(x=..., y=...)` — but do not do that here. The two
functions use different labels: `pearsonr` calls them `x` and `y`, while `spearmanr` calls them `a`
and `b`, so `spearmanr(x=..., y=...)` fails with an error. Giving them in order works for both.

Note that the p-values here are so small that Python prints them in scientific notation:
`5.9998e-21` means 20 zeros after the decimal point before the first digit.


In [ ]:
r, p_value = pearsonr(clean_df['Population Urban %'], clean_df['Life Expectancy Female'])
print('Pearson  r =', round(r, 3), '  p =', p_value)

In [ ]:
rho, p_value = spearmanr(clean_df['Population Urban %'], clean_df['Life Expectancy Female'])
print('Spearman rho =', round(rho, 3), '  p =', p_value)

Step 1 is the chart. Always.


In [ ]:
sns.scatterplot(data=clean_df, x='Population Urban %', y='Life Expectancy Female', alpha=0.6)
plt.show()

## 1.3 · The second pair: health spending and how long women live

Same three steps, different column. Fill in the call — the skew rule to choose Pearson or Spearman (see above) has already told you which
coefficient you are supposed to report, but run **both**, because the gap between them is the
question.

**Canvas question 1** asks which one the rule gives you.


In [ ]:
r, p_value = pearsonr(clean_df['Health Exp/Capita'], clean_df['Life Expectancy Female'])
print('Pearson  r =', round(r, 3), '  p =', p_value)

In [ ]:
rho, p_value = ____(clean_df['Health Exp/Capita'], clean_df['Life Expectancy Female'])
print('Spearman rho =', round(rho, 3), '  p =', p_value)

In [ ]:
sns.scatterplot(data=clean_df, x='Health Exp/Capita', y='Life Expectancy Female', alpha=0.6)
plt.show()

**(a)** Take the two pairs in turn — `Population Urban %` against `Life Expectancy Female`, and
`Health Exp/Capita` against `Life Expectancy Female`.

For each pair, say in a sentence: what you have, what you want, which test to choose (use the skew
rule below if needed), and one thing you can see in the chart that the value generated by the test
you choose does not tell you.

| What you have | Your question | The test | And always draw |
|---|---|---|---|
| two numbers, both symmetric | do they move together? | `pearsonr()` | `scatterplot()` |
| two numbers, one skewed | do they move together? | `spearmanr()` | `scatterplot()` |
| one number, one category | is it different by group? | `f_oneway()` | `boxplot(x=cat, y=num)` |
| two categories | are the two related? | `chi2_contingency()` | `countplot(x=, hue=)` |

**The skew rule.** Calculate skew first, decide which test to use second. If either column fails the
check in the left-hand column, the pair fails.

| Check | `pearsonr()` or `spearmanr()` |
|---|---|
| between −1 and 1 | close enough to symmetric → `pearsonr()` |
| outside −1 and 1 | skewed → `spearmanr()` |

> ✏️ **Answer in Canvas** — *Week 3 Homework - Relationships Between Variables*, question **(a)**.


**(b)** On the second pair, Pearson gives **+0.524** and Spearman gives **+0.849** — they
are a long way apart.

Explain what that gap is telling you. Then answer the question a colleague will actually ask:
*"which one is right?"*

*Four or five sentences.*

> ✏️ **Answer in Canvas** — *Week 3 Homework - Relationships Between Variables*, question **(b)**.


---
# Part 2 · A number across groups

## 2.1 · Making a category out of a number

There is only one text column in this table, so to compare a number across groups you need a second
category. `pd.cut()` makes one by slicing a numeric column into bands.

The cell is given. Two things worth reading in it: the first cut point in `bins` is **0**, which is below the
smallest value in the column — if it were not, the smallest country would silently become `NaN` — and `labels` are names given to each bin which will be the values in this new column `spend_band`.

*(`pd.cut()` is on the Reference slides at the back of Tuesday's deck.)*


In [ ]:
clean_df = clean_df.copy()
clean_df['spend_band'] = pd.cut(clean_df['Health Exp/Capita'],
                                bins=[0, 76, 500, 1200, 8695],
                                labels=['Very Low', 'Low', 'Medium', 'High'])

clean_df['spend_band'].value_counts()

## 2.2 · The test

Splitting a column by a category is the row-selection form from class: *from `clean_df`, take the
rows where the band is 'Very Low', for example, then take the life expectancy column only.*


In [ ]:
very_low = clean_df[clean_df['spend_band'] == 'Very Low']['Life Expectancy Female']
low      = clean_df[clean_df['spend_band'] == 'Low']['Life Expectancy Female']
medium   = clean_df[clean_df['spend_band'] == 'Medium']['Life Expectancy Female']
high     = clean_df[clean_df['spend_band'] == 'High']['Life Expectancy Female']

print('group sizes:', len(very_low), len(low), len(medium), len(high))

`f_oneway()` takes one argument per group. Fill it in.


In [ ]:
F, p_value = f_oneway(____)
print('F =', round(F, 3), '  p =', p_value)

In [ ]:
plt.figure(figsize=(9, 4))
sns.boxplot(data=clean_df, x='spend_band', y='Life Expectancy Female')
plt.show()

## 2.3 · A second test, on a different question

Now a genuinely separate question, and one somebody at the foundation has actually asked: **do the
regions differ in population?**

This part goes back to `world_df`, because `Population (M)` has no missing values and there is no
reason to lose 28 countries for it.

Here is the row-selection form again, written out for one region.


In [ ]:
africa = world_df[world_df['Region'] == 'Africa']['Population (M)']
africa.describe()

Writing that out six times is tedious rather than instructive, so here is a helper that does
it. Read the docstring; you do not need to be able to write this.


In [ ]:
def groups_by(df, category, number):
    """Split `number` into one Series per level of `category`, dropping rows with gaps.

    Returns a list of Series, ready to hand straight to f_oneway(*groups).
    """
    clean = df[[category, number]].dropna()
    levels = sorted(clean[category].unique())
    return [clean[clean[category] == level][number] for level in levels]

In [ ]:
pop_groups = groups_by(world_df, 'Region', 'Population (M)')
F, p_value = f_oneway(*pop_groups)
print('Population (M) by Region:   F =', round(F, 3), '  p =', round(p_value, 4))

That p-value is under 0.05. Before you write anything down, do step 1 — draw it.


In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(data=world_df, x='Region', y='Population (M)')
plt.show()

And look at the numbers behind the boxes. The `std` column is the standard deviation — how spread out
the countries are *inside* each region. You will need it for part 3 of question (c).


In [ ]:
world_df.pivot_table(index='Region', values='Population (M)',
                     aggfunc=['median', 'mean', 'std', 'max', 'count'])

One more number before you decide what to say. This is the column whose skewness you
computed in the Week 1 homework.


In [ ]:
world_df['Population (M)'].skew()

## 2.4 · A third test, for contrast

One more, on a question with a different kind of answer. **Canvas question 4** asks what you may
report from it.


In [ ]:
days_groups = groups_by(world_df, 'Region', 'Days to Start Business')

F, p_value = f_oneway(*days_groups)
print('Days to Start Business by Region:   F =', round(F, 3), '  p =', round(p_value, 4))

## 2.5 · How much does that verdict rest on?

The population test came back significant. Before you write it down, ask how much of it is coming
from a handful of rows. The cell below runs the **same test on `Population (M)`** four times, taking
out the largest countries one at a time.

Read all four lines before you decide what the first one meant.


In [ ]:
def population_anova(label, drop=()):
    kept = world_df[~world_df['Country'].isin(drop)]
    F, p_value = f_oneway(*groups_by(kept, 'Region', 'Population (M)'))
    print(f"{label:34} F = {F:6.3f}   p = {p_value:.4f}")

population_anova('all 208 countries')
population_anova('without China',                    ['China'])
population_anova('without China and India',          ['China', 'India'])
population_anova('without China, India and the US',  ['China', 'India', 'United States'])

**(c)** A colleague wants to put this sentence in the foundation's annual report:

> *"Population differs significantly by region according to ANOVA (p = 0.007)."*

**The test does support that sentence.** ANOVA compares group means, the means are far apart, and the
p-value is real. The question is not whether the sentence is true — it is whether it is the right
sentence, resting on the right evidence.

**One or two sentences for each of the five.**

1. ANOVA compares **means**, and nothing else. Using the boxplot, the table, and the skewness of
   `Population (M)`, say why the mean is a poor summary of this particular column.
2. ANOVA also assumes the groups have **comparable spread**. Look at the `std` column. Is that
   assumption met here, and what does your answer mean for the p-value in the sentence?
3. A board member reads that sentence and concludes that the typical country differs from region to
   region. **Is that conclusion right?** Say which numbers in your output settle it — and whether they
   are the numbers the sentence cites.
4. Section 2.5 re-runs the same test with the largest countries removed one at a time. Report what
   happens to the p-value, and say what that tells you about the original result. *This is a check on
   how much a conclusion depends on a few rows — not a licence to delete rows until you like the
   answer.*
5. Write the sentence you would put in the report instead.

> ✏️ **Answer in Canvas** — *Week 3 Homework - Relationships Between Variables*, question **(c)**.


---
# Part 3 · Two categories

Both `Region` and `spend_band` are categories, so the question *are these two related?* takes the
third test. The table you run it on is a cross-tab; the chart that goes with it is a `countplot`
split by `hue`.

Fill in the two columns.


In [ ]:
table = pd.crosstab(clean_df['____'], clean_df['____'])
table

Two of those cells are **0**. Every one of the 180 countries in `clean_df` has a region and
a band, so nothing is missing — the zeros are facts about the world.

Now the test. Note that it hands back **four** things, and that the one everybody throws away is the
useful one.


In [ ]:
chi2, p_value, dof, expected = chi2_contingency(table)
print('chi2 =', round(chi2, 3), '  p =', p_value, '  dof =', dof)

`expected` is what the table would look like if region and spending band had nothing to do
with each other. The convention is that every expected count should be at least 5. **Canvas
question 3** asks how many of them are not.


In [ ]:
print(expected.round(2))
print()
print('cells with an expected count below 5:', (expected < 5).sum(), 'of', expected.size)
print('smallest expected count:', round(expected.min(), 3))

And the chart. `countplot` with `hue=` is the two-category chart.


In [ ]:
plt.figure(figsize=(10, 4))
sns.countplot(data=clean_df, x='Region', hue='spend_band')
plt.show()

**(d)** Your chi-square returned **χ² = 123.0** with a p-value of about
**5 × 10⁻¹⁹**, and **8 of the 24 expected counts are below 5** — the smallest is **1.67**.

1. In one sentence: what does that p-value let you claim about `Region` and spending band, and what
   does it **not** let you claim? Set the expected counts aside for this part — part 2 comes back to
   them.
2. Look at the `expected` array. The eight small cells are not scattered — they fill two whole rows.
   Name those two regions and say what they have in common that puts them there.
3. Write the **footnote** you would put next to the claim if it went into the foundation's report.
   One or two sentences to warn the reader about the limitations.

> ✏️ **Answer in Canvas** — *Week 3 Homework - Relationships Between Variables*, question **(d)**.


---
# Part 4 · The Analyst's Note

**About 150 words. This is worth a meaningful share of the grade.**

A health ministry has read the foundation's briefing and written back with one question:

> *"Our finance minister wants a number. If we double what we spend per person on health, how many
> years of life expectancy do we buy?"*

Write the reply.

A good note:

- **answers the question that was asked** before explaining why it is the wrong question
- quotes **numbers from your own output**
- says what would change the answer — the thing you would need to know about this particular country
  before you could say anything useful
- does not use the words *correlation*, *significant*, *p-value*, *Spearman* or *skewed*

That last constraint is not a style exercise. A finance minister is going to act on one sentence of
this, and every one of those words invites a follow-up question you do not want to be answering in
that meeting.


> ✏️ **Answer in Canvas** — *Week 3 Homework - Relationships Between Variables*, question **The Analyst's Note**.


---
### Before you submit

- [ ] **Runtime → Restart session and run all.** It must run top to bottom with no errors.
- [ ] Every `____` filled in.
- [ ] All nine answers submitted in the Canvas quiz **Week 3 Homework - Relationships Between Variables** — the four checks, (a), (b),
      (c), (d), and the note.
- [ ] This notebook shared — **Share → General access → Anyone with the link** — and its link pasted
      into the quiz's first question. Check it opens in a private window.
- [ ] Any cells an AI left behind, out of order or duplicated, cleaned up.
- [ ] Your name in the filename.
